In [73]:
import numpy as np
import torch
import pandas as pd
from torch import nn,optim
from torch.utils.data import DataLoader,Dataset, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

## Data

In [5]:
dataset = pd.read_csv('./diabetes.csv')
dataset.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
num_samples , num_features = dataset.shape

In [ ]:
X = dataset.drop(['Outcome'], axis=1).values
X.shape

(768, 8)

In [12]:
y = dataset['Outcome'].to_numpy().reshape(-1, 1)
y.shape

(768, 1)

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=12)
X_train.shape , X_test.shape , y_train.shape, y_test.shape

((614, 8), (154, 8), (614, 1), (154, 1))

In [19]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train,y_train,test_size=0.3,random_state=12)
X_train.shape,  X_valid.shape, y_train.shape, y_valid.shape

((429, 8), (185, 8), (429, 1), (185, 1))

In [20]:
X_scaler = StandardScaler()

X_train = X_scaler.fit_transform(X_train)
X_valid = X_scaler.transform(X_valid)
X_test = X_scaler.transform(X_test)

X_train.shape,  X_valid.shape, X_test.shape, y_train.shape, y_valid.shape, y_test.shape

((429, 8), (185, 8), (154, 8), (429, 1), (185, 1), (154, 1))

In [ ]:
X_train = torch.tensor(X_train,dtype=torch.float32)
X_valid = torch.tensor(X_valid,dtype=torch.float32)
X_test = torch.tensor(X_test,dtype=torch.float32)
y_train = torch.tensor(y_train,dtype=torch.float32)
y_valid = torch.tensor(y_valid,dtype=torch.float32)
y_test = torch.tensor(y_test,dtype=torch.float32)
X_train.shape,  X_valid.shape, X_test.shape, y_train.shape, y_valid.shape, y_test.shape

(torch.Size([429, 8]),
 torch.Size([185, 8]),
 torch.Size([154, 8]),
 torch.Size([429, 1]),
 torch.Size([185, 1]),
 torch.Size([154, 1]))

In [23]:
train_dataset = TensorDataset(X_train,y_train)
valid_dataset = TensorDataset(X_valid,y_valid)
test_dataset = TensorDataset(X_test,y_test)

In [24]:
train_loader = DataLoader(train_dataset,batch_size=100,shuffle=True)
valid_loader = DataLoader(valid_dataset,batch_size=50,shuffle=False)
test_loader = DataLoader(test_dataset,batch_size=154,shuffle=False)

## Model

In [63]:
model = nn.Sequential(
    nn.Linear((num_features-1),1),
    nn.Sigmoid())

In [67]:
loss_fn = nn.BCELoss()
loss_fn

BCELoss()

In [70]:
optimizer = optim.SGD(model.parameters(),0.1,momentum=0.9)

In [71]:
loss_train_hist, loss_valid_hist =[] , []
acc_train_hist, acc_valid_hist =[] , []

In [86]:
n_epoches = 10

for epoch in range(n_epoches):
    mean_loss_train, mean_loss_valid = [], []
    mean_acc_train, mean_acc_valid = 0, 0

    for X_batch, y_batch in train_loader:
        y_hat = model(X_batch)
        loss = loss_fn(y_hat, y_batch)
        mean_loss_train.append(loss.item())

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        mean_acc_train += torch.sum(y_hat.round() == y_batch).item() / len(train_dataset)

    loss_train_hist.append(torch.mean(torch.tensor(mean_loss_train)).item())
    acc_train_hist.append(mean_acc_train)

    print(f'epoch : {epoch} train loss : {loss_train_hist[-1]}\tAccuracy : {mean_acc_train}')

    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            y_hat = model(X_batch)
            loss = loss_fn(y_hat, y_batch)
            mean_loss_valid.append(loss.item())
            mean_acc_valid += torch.sum(y_hat.round() == y_batch).item() / len(valid_dataset)

        loss_valid_hist.append(torch.mean(torch.tensor(mean_loss_valid)).item())
        acc_valid_hist.append(mean_acc_valid)

    print(f'validation loss : {loss_valid_hist[-1]}\tAccuracy : {mean_acc_valid}')

epoch : 0 train loss : 0.4502085745334625	Accuracy : 0.7832167832167832
validation loss : 0.5218670964241028	Accuracy : 0.745945945945946
epoch : 1 train loss : 0.42693910002708435	Accuracy : 0.7855477855477855
validation loss : 0.52195805311203	Accuracy : 0.7513513513513514
epoch : 2 train loss : 0.44280949234962463	Accuracy : 0.7855477855477854
validation loss : 0.5208298563957214	Accuracy : 0.7513513513513514
epoch : 3 train loss : 0.4255855083465576	Accuracy : 0.7785547785547786
validation loss : 0.5196254849433899	Accuracy : 0.7405405405405405
epoch : 4 train loss : 0.447927325963974	Accuracy : 0.7738927738927739
validation loss : 0.5194153189659119	Accuracy : 0.745945945945946
epoch : 5 train loss : 0.43464335799217224	Accuracy : 0.7762237762237761
validation loss : 0.5199005603790283	Accuracy : 0.7567567567567568
epoch : 6 train loss : 0.4249454438686371	Accuracy : 0.7785547785547786
validation loss : 0.5212063789367676	Accuracy : 0.7405405405405405
epoch : 7 train loss : 0.4718

In [92]:
mean_acc_test = 0
with torch.no_grad():
    for X_batch,y_batch in test_loader:
        y_hat = model(X_batch)
        error = nn.functional.l1_loss(y_hat,y_batch)
        print(error)

        mean_acc_test += torch.sum(y_hat.round() == y_batch).item() / len(test_dataset)
        print(mean_acc_test)

tensor(0.2901)
0.8116883116883117
